# Import libraries

In [ ]:
from google.cloud import bigquery

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv


# Find the private .env file from either the Notebook folder
# or the project root.
env_path = Path("../.env")

if not env_path.exists():
    env_path = Path(".env")

load_dotenv(env_path)

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET_ID = os.environ["GCP_RAW_DATASET"]

client = bigquery.Client(project=PROJECT_ID)

tables = client.list_tables(f"{PROJECT_ID}.{DATASET_ID}")

print("BigQuery connection successful.")

for table in tables:
    print(table.table_id)

In [ ]:
aisles = client.query(
    f"""
    SELECT
        aisle_id,
        aisle
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_aisles`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print("Aisles loaded from BigQuery:", aisles.shape)
aisles.head()

In [ ]:
departments = client.query(
    f"""
    SELECT
        department_id,
        department
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_departments`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print(
    "Departments loaded from BigQuery:",
    departments.shape,
)

departments.head()

In [ ]:
products = client.query(
    f"""
    SELECT
        product_id,
        product_name,
        aisle_id,
        department_id
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_products`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print("Products loaded from BigQuery:", products.shape)
products.head()

In [ ]:
orders = client.query(
    f"""
    SELECT
        order_id,
        user_id,
        eval_set,
        order_number,
        order_dow,
        order_hour_of_day,
        days_since_prior_order
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_orders`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print("Orders loaded from BigQuery:", orders.shape)
orders.head()

In [ ]:
order_products_train = client.query(
    f"""
    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        reordered
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_train`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print(
    "Train order products loaded from BigQuery:",
    order_products_train.shape,
)

order_products_train.head()

In [ ]:
order_products_prior = client.query(
    f"""
    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        reordered
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_prior`
    """
).to_dataframe(
    create_bqstorage_client=True,
    progress_bar_type="tqdm",
)

print(
    "Prior order products loaded from BigQuery:",
    order_products_prior.shape,
)

order_products_prior.head()

# Set the Dataset folder path

In [ ]:
# The notebook is inside the Notebook folder
data_dir = Path("../Dataset")

# Fallback if VS Code starts from the project root
if not data_dir.exists():
    data_dir = Path("Dataset")

print("Dataset folder:", data_dir.resolve())

In [ ]:
# Validate row counts directly inside BigQuery.
# Only the six summary results are downloaded into Pandas.

bigquery_row_counts = client.query(
    f"""
    SELECT 'aisles' AS table_name, COUNT(*) AS row_count
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_aisles`

    UNION ALL

    SELECT 'departments', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_departments`

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_products`

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_orders`

    UNION ALL

    SELECT 'order_products_prior', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_prior`

    UNION ALL

    SELECT 'order_products_train', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_train`
    """
).to_dataframe(create_bqstorage_client=False)

bigquery_row_counts

# Load all six CSV files

In [ ]:
#aisles = pd.read_csv(data_dir / "aisles.csv")
#departments = pd.read_csv(data_dir / "departments.csv")
#orders = pd.read_csv(data_dir / "orders.csv")
#products = pd.read_csv(data_dir / "products.csv")

#order_products_prior = pd.read_csv(
#    data_dir / "order_products__prior.csv"
#)

#order_products_train = pd.read_csv(
#    data_dir / "order_products__train.csv"
#)

In [ ]:
# Verify that the downloaded Pandas tables match BigQuery

loaded_tables = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
}

expected_counts = dict(
    zip(
        bigquery_row_counts["table_name"],
        bigquery_row_counts["row_count"],
    )
)

print("Source: BigQuery raw dataset configured through .env\n")


all_passed = True

for table_name, dataframe in loaded_tables.items():
    bigquery_count = int(expected_counts[table_name])
    downloaded_count = len(dataframe)
    status = "PASS" if downloaded_count == bigquery_count else "FAIL"

    if status == "FAIL":
        all_passed = False

    print(
        f"{status}: {table_name} — "
        f"BigQuery: {bigquery_count:,}, "
        f"Pandas: {downloaded_count:,}"
    )

if all_passed:
    print("\nAll six tables were downloaded successfully from BigQuery.")
else:
    print("\nOne or more downloaded tables do not match BigQuery.")

In [ ]:
# Verify that all six Pandas DataFrames came from BigQuery

bigquery_row_counts = client.query(
    f"""
    SELECT 'aisles' AS table_name, COUNT(*) AS row_count
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_aisles`

    UNION ALL

    SELECT 'departments', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_departments`

    UNION ALL

    SELECT 'products', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_products`

    UNION ALL

    SELECT 'orders', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_orders`

    UNION ALL

    SELECT 'order_products_prior', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_prior`

    UNION ALL

    SELECT 'order_products_train', COUNT(*)
    FROM `{PROJECT_ID}.{DATASET_ID}.raw_order_products_train`
    """
).to_dataframe(create_bqstorage_client=False)


loaded_tables = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
}


expected_counts = dict(
    zip(
        bigquery_row_counts["table_name"],
        bigquery_row_counts["row_count"],
    )
)


print(f"Data source: BigQuery `{PROJECT_ID}.{DATASET_ID}`\n")

all_passed = True

for table_name, dataframe in loaded_tables.items():
    bigquery_count = int(expected_counts[table_name])
    pandas_count = len(dataframe)

    if pandas_count == bigquery_count:
        status = "PASS"
    else:
        status = "FAIL"
        all_passed = False

    print(
        f"{status}: {table_name} — "
        f"BigQuery rows: {bigquery_count:,}; "
        f"Pandas rows: {pandas_count:,}"
    )


if all_passed:
    print(
        "\nPASS: All six datasets were downloaded "
        "successfully from BigQuery."
    )
else:
    print(
        "\nFAIL: One or more Pandas DataFrames "
        "do not match BigQuery."
    )

# Check all loaded tables

In [ ]:
tables = {
    "aisles": aisles,
    "departments": departments,
    "orders": orders,
    "products": products,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train
}

for table_name, df in tables.items():
    print(f"{table_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Preview one table

In [ ]:
orders.head()

In [ ]:
products.head()


In [ ]:
aisles.head()


In [ ]:
departments.head()


In [ ]:
order_products_prior.head()


In [ ]:
order_products_train.head()

# View the first five rows

In [ ]:
for table_name, df in tables.items():
    print(f"\n===== {table_name.upper()} =====")
    display(df.head())

# Check rows and columns

In [ ]:
shape_summary = pd.DataFrame({
    table_name: {
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for table_name, df in tables.items()
}).T

shape_summary


# Check column information

In [ ]:
print("Missing values in aisles.csv")
aisles.isnull().sum()

In [ ]:
print("Missing values in departments.csv")
departments.isnull().sum()


In [ ]:
print("Missing values in orders.csv")
orders.isnull().sum()

In [ ]:
print("Missing values in products.csv")
products.isnull().sum()

In [ ]:
print("Missing values in order_products__prior.csv")
order_products_prior.isnull().sum()

In [ ]:
print("Missing values in order_products__train.csv")
order_products_train.isnull().sum()

In [ ]:
missing = aisles.isnull().sum()

pd.DataFrame({
    "Column": aisles.columns,
    "Data Type": aisles.dtypes.astype(str).values,
    "Total Rows": len(aisles),
    "Non-Null Values": len(aisles) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (missing.values / len(aisles) * 100).round(2)
})

In [ ]:
missing = departments.isnull().sum()

pd.DataFrame({
    "Column": departments.columns,
    "Data Type": departments.dtypes.astype(str).values,
    "Total Rows": len(departments),
    "Non-Null Values": len(departments) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (missing.values / len(departments) * 100).round(2)
})

In [ ]:
missing = orders.isnull().sum()

pd.DataFrame({
    "Column": orders.columns,
    "Data Type": orders.dtypes.astype(str).values,
    "Total Rows": len(orders),
    "Non-Null Values": len(orders) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (missing.values / len(orders) * 100).round(2)
})

In [ ]:
missing = products.isnull().sum()

pd.DataFrame({
    "Column": products.columns,
    "Data Type": products.dtypes.astype(str).values,
    "Total Rows": len(products),
    "Non-Null Values": len(products) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (missing.values / len(products) * 100).round(2)
})

In [ ]:
missing = order_products_prior.isnull().sum()

pd.DataFrame({
    "Column": order_products_prior.columns,
    "Data Type": order_products_prior.dtypes.astype(str).values,
    "Total Rows": len(order_products_prior),
    "Non-Null Values": len(order_products_prior) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (
        missing.values / len(order_products_prior) * 100
    ).round(2)
})

In [ ]:
missing = order_products_train.isnull().sum()

pd.DataFrame({
    "Column": order_products_train.columns,
    "Data Type": order_products_train.dtypes.astype(str).values,
    "Total Rows": len(order_products_train),
    "Non-Null Values": len(order_products_train) - missing.values,
    "Missing Values": missing.values,
    "Missing (%)": (
        missing.values / len(order_products_train) * 100
    ).round(2)
})

In [ ]:
print("Description of aisles.csv")
display(aisles.describe(include="all").T)

In [ ]:
print("Description of departments.csv")
display(departments.describe(include="all").T)

In [ ]:
print("Description of orders.csv")
display(orders.describe(include="all").T)

In [ ]:
print("Description of products.csv")
display(products.describe(include="all").T)

In [ ]:
print("Description of order_products__prior.csv")
display(order_products_prior.describe(include="all").T)

In [ ]:
print("Description of order_products__train.csv")
display(order_products_train.describe(include="all").T)

In [ ]:
print("Exact duplicate rows:", aisles.duplicated().sum())
print(
    "Duplicate aisle_id:",
    aisles.duplicated(subset=["aisle_id"]).sum()
)

In [ ]:
print("Exact duplicate rows:", departments.duplicated().sum())
print(
    "Duplicate department_id:",
    departments.duplicated(subset=["department_id"]).sum()
)

In [ ]:
print("Exact duplicate rows:", orders.duplicated().sum())
print(
    "Duplicate order_id:",
    orders.duplicated(subset=["order_id"]).sum()
)

In [ ]:
print("Exact duplicate rows:", products.duplicated().sum())
print(
    "Duplicate product_id:",
    products.duplicated(subset=["product_id"]).sum()
)

In [ ]:
print(
    "Exact duplicate rows:",
    order_products_prior.duplicated().sum()
)

print(
    "Duplicate order-product combinations:",
    order_products_prior.duplicated(
        subset=["order_id", "product_id"]
    ).sum()
)

In [ ]:
print(
    "Exact duplicate rows:",
    order_products_train.duplicated().sum()
)

print(
    "Duplicate order-product combinations:",
    order_products_train.duplicated(
        subset=["order_id", "product_id"]
    ).sum()
)

### Blank values in days_since_prior_order were converted to numeric NaN. These values represent customers’ first orders, where no previous-order interval exists. They were retained to preserve valid records and avoid incorrectly interpreting them as zero-day reorders.

In [ ]:
orders["days_since_prior_order"] = pd.to_numeric(
    orders["days_since_prior_order"],
    errors="coerce"
)

# Check Outlier

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### orders.csv — order_number

In [ ]:
Q1 = orders["order_number"].quantile(0.25)
Q3 = orders["order_number"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

order_number_outliers = orders[
    (orders["order_number"] < lower) |
    (orders["order_number"] > upper)
]

print("Lower limit:", lower)
print("Upper limit:", upper)
print("Number of outliers:", len(order_number_outliers))

display(order_number_outliers)

plt.boxplot(orders["order_number"], vert=False)
plt.title("Outliers in Order Number")
plt.xlabel("Order Number")
plt.show()

### orders.csv — days_since_prior_order

In [ ]:
days = orders["days_since_prior_order"].dropna()

Q1 = days.quantile(0.25)
Q3 = days.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

days_outliers = orders[
    (orders["days_since_prior_order"] < lower) |
    (orders["days_since_prior_order"] > upper)
]

print("Lower limit:", lower)
print("Upper limit:", upper)
print("Number of outliers:", len(days_outliers))

display(days_outliers)

plt.boxplot(days, vert=False)
plt.title("Outliers in Days Since Prior Order")
plt.xlabel("Days")
plt.show()

### order_products__prior.csv — add_to_cart_order

In [ ]:
cart = order_products_prior["add_to_cart_order"]

Q1 = cart.quantile(0.25)
Q3 = cart.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

prior_outliers = order_products_prior[
    (cart < lower) | (cart > upper)
]

print("Number of outliers:", len(prior_outliers))
display(prior_outliers)

plt.boxplot(cart, vert=False)
plt.title("Outliers in order_products__prior.csv")
plt.xlabel("Add-to-Cart Order")
plt.show()

### order_products__train.csv  — add_to_cart_order

In [ ]:
cart = order_products_train["add_to_cart_order"]

Q1 = cart.quantile(0.25)
Q3 = cart.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

train_outliers = order_products_train[
    (cart < lower) | (cart > upper)
]

print("Number of outliers:", len(train_outliers))
display(train_outliers)

plt.boxplot(cart, vert=False)
plt.title("Outliers in order_products__train.csv")
plt.xlabel("Add-to-Cart Order")
plt.show()

### three graphs show potential IQR outliers. They are the circles beyond the right whisker.

### Variable	Outlier rule	Count	Interpretation
### order_number	Above 50	216,870	Frequent customers with many orders
### Prior add_to_cart_order	Above 23	1,357,124	Products in large baskets
### Train add_to_cart_order	26 or above	50,853	Products in large baskets

### These are unusual values, but not data errors. Keep them because frequent customers and large baskets are useful for market-basket and cross-selling analysis. Do not remove them automatically.

In [ ]:
print(len(order_number_outliers))
print(len(prior_outliers))
print(len(train_outliers))

### aisles, departments, and products contain identifiers/categories, so IQR outlier detection is not meaningful.
### orders passed all valid-range checks.
### order_products__prior: 1,357,124 potential outliers (4.18%), with cart positions above 23.
### order_products__train: 50,853 potential outliers (3.67%), with cart positions above 25.5.
### These are probably valid large shopping baskets, so they should not be automatically removed.


In [ ]:
# Convert BigQuery text fields into correct numeric data types

integer_columns = {
    "aisles": {
        "dataframe": aisles,
        "columns": ["aisle_id"],
    },
    "departments": {
        "dataframe": departments,
        "columns": ["department_id"],
    },
    "products": {
        "dataframe": products,
        "columns": [
            "product_id",
            "aisle_id",
            "department_id",
        ],
    },
    "orders": {
        "dataframe": orders,
        "columns": [
            "order_id",
            "user_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
        ],
    },
    "order_products_prior": {
        "dataframe": order_products_prior,
        "columns": [
            "order_id",
            "product_id",
            "add_to_cart_order",
            "reordered",
        ],
    },
    "order_products_train": {
        "dataframe": order_products_train,
        "columns": [
            "order_id",
            "product_id",
            "add_to_cart_order",
            "reordered",
        ],
    },
}


for table_name, specification in integer_columns.items():
    dataframe = specification["dataframe"]

    for column in specification["columns"]:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="raise",
        ).astype("int64")

    print(f"PASS: {table_name} numeric columns converted")


# Preserve meaningful missing values for first orders
orders["days_since_prior_order"] = pd.to_numeric(
    orders["days_since_prior_order"],
    errors="coerce",
).astype("float64")

print(
    "PASS: days_since_prior_order converted; "
    "first-order NaN values retained"
)

In [ ]:
# Export validated datasets for the Feature Engineering notebook

cleaned_dir = data_dir / "cleaned"
cleaned_dir.mkdir(parents=True, exist_ok=True)

cleaned_tables = {
    "aisles": aisles,
    "departments": departments,
    "products": products,
    "orders": orders,
    "order_products_prior": order_products_prior,
    "order_products_train": order_products_train,
}

for table_name, dataframe in cleaned_tables.items():
    output_path = cleaned_dir / f"{table_name}.parquet"

    dataframe.to_parquet(
        output_path,
        index=False,
    )

    print(
        f"PASS: {table_name} — "
        f"{len(dataframe):,} rows saved to {output_path}"
    )

print(
    "\nData lineage: BigQuery raw tables "
    "→ Data Cleaning "
    "→ cleaned Parquet files"
)
print("Feature Engineering inputs are ready.")